# Figure 4 counterfactual response

Replots the main-text counterfactual diagnostics from compact exported summary data. Outputs are written to `logs/scripts/figure4_counterfactual/`.

In [1]:
import os
from pathlib import Path
if Path.cwd().name == "scripts":
    os.chdir(Path.cwd().parent)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

save_path = Path("logs/scripts/figure4_counterfactual")
save_path.mkdir(parents=True, exist_ok=True)
metrics = pd.read_csv("data/counterfactual/main_figure_counterfactual_metrics.csv")
metrics.head()


,metric,model,detail,phase,value,ci_low,ci_high
0,Duration,CESM2,Ablation,Ramp-Down,5.051500,4.828500,5.274600
1,Duration,CESM2,Ablation,Ramp-Up,5.179300,5.018600,5.340100
2,Duration,CESM2,Baseline,Ramp-Down,3.474100,3.309100,3.639200
3,Duration,CESM2,Baseline,Ramp-Up,3.459100,3.317700,3.600600
4,Duration,CESM2,CMIP6,Ramp-Down,2.782609,2.782609,2.782609


In [2]:
# Panels a-c: the exact MIROC-ES2L values and 95% confidence intervals
# produced by 260609反事实实验结果汇总.ipynb.
metric_specs = {
    'Variance': ('Fig4a', 'ENSO Index Variance (°C²)'),
    'Super-Event': ('Fig4b', 'Super El Niño Ratio (>2.0°C)'),
    'Duration': ('Fig4c', 'ENSO Index Period (year)'),
}
order = ['CMIP6', 'Ours', 'Baseline', 'Ablation']
labels = ['CMIP6\n(MIROC-ES2L)', 'Ours', 'XRO', 'Ours (w/o CO₂)']
phases = ['Ramp-Up', 'Ramp-Down']
colors = {'Ramp-Up': '#c44569', 'Ramp-Down': '#546de5'}
for metric, (filename, ylabel) in metric_specs.items():
    part = metrics.query("metric == @metric and model == 'MIROC-ES2L'").set_index(['detail', 'phase'])
    fig, ax = plt.subplots(figsize=(7.2, 4.6), dpi=200)
    x = np.arange(len(order)); width = 0.36
    for j, phase in enumerate(phases):
        values = np.array([part.loc[(detail, phase), 'value'] for detail in order])
        low = np.array([part.loc[(detail, phase), 'ci_low'] for detail in order])
        high = np.array([part.loc[(detail, phase), 'ci_high'] for detail in order])
        ax.bar(x + (j-.5)*width, values, width, color=colors[phase], label=phase.replace('-', '-').title() + ' Phase',
               yerr=np.vstack([values-low, high-values]), error_kw=dict(ecolor='#3d3d3d', lw=1, capsize=4, capthick=1))
    ax.set_xticks(x, labels)
    ax.set_ylabel(ylabel)
    if metric == 'Super-Event':
        ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
    ax.legend(loc='lower left', frameon=True)
    fig.tight_layout()
    for suffix in ('png', 'svg', 'pdf'):
        fig.savefig(save_path / f'{filename}.{suffix}', dpi=300, transparent=True)
    plt.close(fig)


In [3]:
# Panel d: the original barplot aggregates ramp-down-minus-ramp-up changes
# across the diagnostic-specific four-model subsets.
diff = metrics.pivot(index=['metric', 'model', 'detail'], columns='phase', values='value').reset_index()
diff['Difference'] = diff['Ramp-Down'] - diff['Ramp-Up']
fig, axes = plt.subplots(1, 3, figsize=(9.0, 4.2), dpi=200)
palette = {'CMIP6':'#546de5', 'Ours':'#c44569', 'Baseline':'#55A868', 'Ablation':'#8172B2'}
for ax, metric, title in zip(axes, ['Variance','Super-Event','Duration'], ['Variance (n=4)','Super-Event (n=4)','Period (n=4)']):
    part = diff.query('metric == @metric')
    means = part.groupby('detail')['Difference'].mean().reindex(order)
    # The source notebook used seaborn's 95% bootstrap CI across four models.
    lo=[]; hi=[]
    rng=np.random.default_rng(0)
    for detail in order:
        v=part.loc[part.detail.eq(detail),'Difference'].to_numpy()
        boot=np.mean(rng.choice(v,(10000,len(v)),replace=True),axis=1)
        a,b=np.percentile(boot,[2.5,97.5]);lo.append(means[detail]-a);hi.append(b-means[detail])
    xx=np.arange(4)
    ax.bar(xx, means, color=[palette[x] for x in order], yerr=np.vstack([lo,hi]),
           error_kw=dict(ecolor='#3d3d3d',lw=1,capsize=3,capthick=1))
    ax.axhline(0,color='0.5',ls='--',lw=.8)
    ax.set_xticks(xx,['CMIP6','Ours','Baseline','Ablation'],rotation=45,ha='right')
    ax.set_title(title); ax.set_xlabel('')
axes[0].set_ylabel('Difference (Ramp-Down − Ramp-Up)')
fig.tight_layout()
for suffix in ('png','svg','pdf'):
    fig.savefig(save_path / f'Fig4d.{suffix}',dpi=300,transparent=True)
plt.close(fig)
